# BPOTF vs MWPM on Surface Codes via Sinter

This notebook benchmarks the BP+BP+OTF decoder against minimum-weight
perfect matching (MWPM via [PyMatching](https://github.com/oscarhiggott/PyMatching))
on rotated surface codes under circuit-level depolarising noise using
[sinter](https://github.com/quantumlib/Stim/tree/main/glue/sample).

The decoder setup follows the same logic as in
`test_scripts/surface_codes/tons_debug.py`:
- The DEM is generated with `decompose_errors=True` and processed with
  `allow_undecomposed_hyperedges=False`.
- The sparsified (edge) check matrix and transfer matrix are obtained
  from the `beliefmatching` library.
- The OTF matrix is built via `otf_matrix_computer`, which adds two
  virtual check rows (one for X-type, one for Z-type boundary edges)
  to the edge check matrix.
- BP iterations are set to [6, 50, 51] for the three stages
  (DEM, phenomenological, OTF).
- Decimation is set to 0.

Reference: deMarti iOlius et al., [arXiv:2409.01440](https://arxiv.org/abs/2409.01440)

In [ ]:
%pip install numpy matplotlib stim sinter scipy

In [ ]:
import sys, os
import numpy as np
import stim
import sinter

from BPOTF import OBPOTF, NoiseType, DemData

sys.path.insert(0, os.path.join("..", "src"))
from sinter_bpotf import SinterBpOtfDecoder

sys.path.insert(0, os.path.join("..", "test_scripts", "surface_codes"))
from otf_matrix import otf_matrix_computer

from ldpc.ckt_noise.dem_matrices import detector_error_model_to_check_matrices

## 1. Define circuit builder

Helper to generate a rotated surface code memory experiment with
uniform depolarising circuit-level noise using stim.

In [ ]:
def make_circuit(d, p):
    """Generate a rotated surface code circuit with circuit-level noise."""
    return stim.Circuit.generated(
        "surface_code:rotated_memory_z",
        distance=d,
        rounds=d,
        after_clifford_depolarization=p,
        after_reset_flip_probability=p,
        before_measure_flip_probability=p,
        before_round_data_depolarization=p,
    )

## 2. Build tasks

Create one sinter task per (distance, error rate) combination. Each task
runs up to 10 000 shots or until 200 logical errors are collected,
whichever comes first.

In [ ]:
distances = [5, 9, 13]
error_rates = [1e-3, 3e-3, 5e-3, 1e-2]

tasks = []
for d in distances:
    for p in error_rates:
        circuit = make_circuit(d, p)
        tasks.append(
            sinter.Task(
                circuit=circuit,
                collection_options=sinter.CollectionOptions(
                    max_shots=10_000,
                    max_errors=200,
                ),
                json_metadata={"d": d, "p": p},
            )
        )

print(f"Created {len(tasks)} tasks")

## 3. Run BPOTF via sinter

Each task has a different DEM (different distance and error rate), so a
`SinterBpOtfDecoder` is built per task. The setup for each decoder:

1. **DEM**: generated with `decompose_errors=True`.
2. **Check matrices**: `check_matrix` (full DEM PCM),
   `edge_check_matrix` (sparsified), and `edge_observables_matrix`
   are extracted via `beliefmatching`.
3. **Transfer matrix**: `hyperedge_to_edge_matrix` maps soft information
   from the full DEM to the sparsified graph.
4. **OTF matrix**: built by `otf_matrix_computer`, which adds two virtual
   check rows to the edge check matrix so that boundary edges (weight 1)
   are properly handled by Kruskal's algorithm.
5. **BP iterations**: [6, 50, 51] for stages 1 (DEM), 2 (phenomenological),
   and 3 (OTF) respectively.
6. **Decimation**: this is the prior of the columns not included in the OTF on the last BP process. It is set to 0.


In [1]:
from ldpc.ckt_noise.dem_matrices import detector_error_model_to_check_matrices

bpotf_samples = []

for task in tasks:
    d = task.json_metadata["d"]
    p = task.json_metadata["p"]
    circuit = task.circuit

    dem = circuit.detector_error_model(decompose_errors=True)
    bm = detector_error_model_to_check_matrices(dem, allow_undecomposed_hyperedges=False)

    phen_check = bm.edge_check_matrix.toarray('F').astype(np.uint8)
    otf_mat = otf_matrix_computer(circuit, phen_check, d).astype(np.uint8)

    decoder = SinterBpOtfDecoder(
        dem=dem,
        p=p,
        pcm=bm.check_matrix,
        obs_matrix=bm.observables_matrix.toarray('F').astype(np.uint8),
        transfer_matrix=bm.hyperedge_to_edge_matrix.toarray().astype(np.uint8),
        phen_check_matrix=phen_check,
        phen_obs_matrix=bm.edge_observables_matrix.toarray('F').astype(np.uint8),
        otf_matrix=otf_mat,
        bp_iters=[6, 50, 51],
        decimation=0,
    )

    result = sinter.collect(
        num_workers=4,
        tasks=[task],
        decoders=["bpotf"],
        custom_decoders={"bpotf": decoder},
    )
    bpotf_samples.extend(result)
    r = result[0]
    print(f"  d={d}, p={p:.4f}: shots={r.shots}, errors={r.errors}")

print(f"\nBPOTF: collected {len(bpotf_samples)} results")

## 4. Run MWPM via sinter

Sinter has built-in support for PyMatching's MWPM decoder via the
`"pymatching"` decoder name. This runs on the same tasks so the results
are directly comparable with BPOTF.

In [ ]:
mwpm_samples = sinter.collect(
    num_workers=4,
    tasks=tasks,
    decoders=["pymatching"],
    print_progress=True,
)

print(f"\nMWPM: collected {len(mwpm_samples)} results")

## 5. Display results side by side

Table comparing the logical error rate per round for BPOTF and MWPM at
each (distance, error rate) point, together with their ratio.

In [ ]:
# Index results by (d, p) for easy lookup
def index_samples(samples):
    result = {}
    for s in samples:
        key = (s.json_metadata["d"], s.json_metadata["p"])
        result[key] = s
    return result

bpotf_idx = index_samples(bpotf_samples)
mwpm_idx = index_samples(mwpm_samples)

print(f"{'d':<5} {'p':<10} {'BPOTF p(e)/d':<18} {'MWPM p(e)/d':<18} {'ratio':<10}")
print("-" * 65)
for d in distances:
    for p in error_rates:
        key = (d, p)
        b = bpotf_idx.get(key)
        m = mwpm_idx.get(key)
        b_rate = b.errors / max(b.shots, 1) / d if b else float('nan')
        m_rate = m.errors / max(m.shots, 1) / d if m else float('nan')
        ratio = b_rate / m_rate if m_rate > 0 else float('nan')
        print(f"{d:<5} {p:<10.4f} {b_rate:<18.6f} {m_rate:<18.6f} {ratio:<10.2f}")

## 6. Plot: BPOTF vs MWPM

Logical error rate per round as a function of the physical error rate.
BPOTF is shown with bold solid lines and circle markers; MWPM with
dashed lines and square markers. Each colour corresponds to a different
code distance.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))

colors = {5: "C0", 9: "C1", 13: "C2"}

for d in distances:
    # BPOTF: bold lines with circles
    b_samples = sorted(
        [s for s in bpotf_samples if s.json_metadata["d"] == d],
        key=lambda s: s.json_metadata["p"],
    )
    b_ps = [s.json_metadata["p"] for s in b_samples]
    b_rates = [s.errors / max(s.shots, 1) / d for s in b_samples]
    ax.plot(b_ps, b_rates, "o-", color=colors[d], linewidth=2, markersize=7,
            label=f"BPOTF d={d}")

    # MWPM: dashed lines with squares
    m_samples = sorted(
        [s for s in mwpm_samples if s.json_metadata["d"] == d],
        key=lambda s: s.json_metadata["p"],
    )
    m_ps = [s.json_metadata["p"] for s in m_samples]
    m_rates = [s.errors / max(s.shots, 1) / d for s in m_samples]
    ax.plot(m_ps, m_rates, "s--", color=colors[d], linewidth=1, markersize=6,
            label=f"MWPM d={d}")

ax.set_xlabel("Physical error rate")
ax.set_ylabel("Logical error rate per round")
ax.set_xscale("log")
ax.set_yscale("log")
ax.legend(ncol=2, fontsize=9)
ax.set_title("BPOTF vs MWPM -- Rotated Surface Code (circuit-level noise)")
ax.grid(True, which="both", ls="--", alpha=0.5)
plt.tight_layout()
plt.show()